In [18]:
import os

from spark_esri import spark_start, spark_stop

In [19]:
spark_stop()

spark = spark_start()

***WARNING*** No active Spark session.


In [20]:
sp_ref = arcpy.SpatialReference(3857)
fields = ["MMSI","SHAPE@X","SHAPE@Y","BaseDateTime"]

In [21]:
with arcpy.da.SearchCursor("Broadcast", fields, spatial_reference=sp_ref) as rows:
    spark\
        .createDataFrame(rows, "mmsi string,x double,y double,t timestamp")\
        .selectExpr("mmsi","x","y","hour(t) h","unix_timestamp(t) t")\
        .createOrReplaceTempView("v0")

In [22]:
prq_path = os.path.join("C:",os.sep,"temp","delete_me.prq")

In [23]:
spark.sql("select avg(x) ax,avg(y) ay from v0")\
    .write\
    .parquet(prq_path, mode="overwrite")

In [24]:
spark\
    .read\
    .parquet(prq_path)\
    .show()

+------------------+------------------+
|                ax|                ay|
+------------------+------------------+
|-8923171.297124285|2972042.3248071168|
+------------------+------------------+

